### Step 1: Environment setup

In [1]:
import os
import json
import time
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("OpenAI key loaded:", "✅" if OPENAI_API_KEY else "❌ Missing!")

missing = [name for name in ("PAGEINDEX_API_KEY", "OPENAI_API_KEY") if not os.getenv(name)]
if missing:
    raise ValueError(f"Set {', '.join(missing)} in your .env file, then rerun this cell.")

PageIndex key loaded: ✅
OpenAI key loaded: ✅


In [3]:
from pageindex import PageIndexClient
from openai import OpenAI

# The current PageIndex SDK reads PAGEINDEX_API_KEY from the environment.
pi_client = PageIndexClient(index="cloud")
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ PageIndex client initialized")
print("✅ OpenAI client initialized")
# Initialization does not verify credentials with the services.

✅ PageIndex client initialized
✅ OpenAI client initialized


### Step 2: Upload and index a PDF

Running the upload cell sends the selected PDF to PageIndex Cloud for processing
and returns a `doc_id` for later operations. Upload success does not mean indexing
has finished.

PageIndex uses a hierarchical document index, so this notebook does not create
fixed-size chunks or a vector database.

In [ ]:
# Locate the project whether the kernel starts in the root or notebook folder.
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "2_RAG" / "data" / "pdf").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start the notebook from inside the LLM_Learning project.")

PDF_PATH = PROJECT_ROOT / "2_RAG" / "data" / "pdf" / "DS_takehome.pdf"

print(f"Uploading: {PDF_PATH}")
# 上传文件，同时发起建索引任务
# 云端接收后会排队、处理并建树；调用返回时，建树可能还没完成
result = pi_client.submit_document(str(PDF_PATH))
doc_id = result["doc_id"]

print("✅ Uploaded; indexing may still be in progress.")
print(f"Document ID: {doc_id}")
print("Save this ID — you will use it throughout the notebook.")


Uploading: /Users/bellahao/Desktop/LLM_Learning/2_RAG/data/pdf/DS_takehome.pdf
✅ Uploaded; indexing may still be in progress.
Document ID: pi-cmu3gzfui000m0cnscgn49yab
Save this ID — you will use it throughout the notebook.


### Wait for indexing to complete

Poll the existing `doc_id` every five seconds. If processing fails, stop with an
error; if it exceeds ten minutes, rerun the polling cell later using the same ID.
You do not need to upload the PDF again.


In [ ]:
POLL_INTERVAL_SECONDS = 5
TIMEOUT_SECONDS = 600

print("⏳ Building tree index...")
deadline = time.monotonic() + TIMEOUT_SECONDS #设定一个 600 秒后的截止点

while True:
    # 询处理进度，不触发建树。上一步已经开始indexing了
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"Status: {status}")

    if status == "completed":
        print("✅ Document processing completed!")
        break
    if status == "failed":
        raise RuntimeError(f"PageIndex processing failed for {doc_id}: {status_result}")
    if status not in {"queued", "processing"}:
        raise RuntimeError(f"Unexpected document status: {status_result}")

    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError(
            f"Processing is still pending after {TIMEOUT_SECONDS} seconds. "
            "Rerun this cell later with the same doc_id."
            #重新跑的话，就是仍然查询同一个 doc_id，
            #只是重新给本地等待计时。如果云端已经完成，第一次查询就会返回 completed
        )
    time.sleep(min(POLL_INTERVAL_SECONDS, remaining))

⏳ Building tree index...
Status: processing
Status: processing
Status: completed
✅ Document processing completed!


### Step 3: Inspect the tree structure

Fetch the tree with node summaries, count its top-level sections, and inspect the
first node as formatted JSON. Nested `nodes` represent subsections.
The full result remains available as `pageindex_tree` (and `tree`).


In [ ]:
# Fetch the full tree with node summaries.这里获取已经生成的树，并请求包含节点摘要
tree_result = pi_client.get_tree(doc_id, node_summary=True)
if tree_result.get("status") != "completed":
    raise RuntimeError(
        f"Tree is not ready: {tree_result.get('status')}. "
        "Run the polling cell before retrying."
    )

pageindex_tree = tree_result["result"]
# 给同一个树对象取另一个名字 tree，方便后面使用。没有复制数据，两个变量指向同一个对象
tree = pageindex_tree  # Keep the existing variable available for later cells.

# 打印顶层章节数量，不包括子章节
print(f"📊 Top-level sections: {len(pageindex_tree)}")

print("\n🌲 Raw tree (first node):")

# 把第一个顶层节点及其包含的子节点以易读的 JSON 格式打印出来：
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2, ensure_ascii=False))


📊 Top-level sections: 28

🌲 Raw tree (first node):
{
  "title": "Preface",
  "node_id": "0000",
  "page_index": 1,
  "summary": "The provided text comprises the title page, author details, and the complete table of contents for 'A Collection of Data Science Take-Home Challenges,' outlining numerous practical case studies and their corresponding solutions, along with promotional links for mentorship and feedback services.",
  "text": "Sold to\n\nQI.TIAN0123@GMAIL.COM\n\n5\n\n\"Best data science job interview resource\"\n\nDataScienceBootcamps.com\n\nA COLLECTION OF\nDATA SCIENCE\nTAKE-HOME CHALLENGES\n\n![img-0.jpeg](img-0.jpeg)\n\nGIULIO PALOMBO\n\n|  Intro | 4  |\n| --- | --- |\n|  Conversion Rate | 5  |\n|  Spanish Translation A/B Test | 7  |\n|  Employee Retention | 11  |\n|  Identifying Fraudulent Activities | 14  |\n|  Funnel Analysis | 18  |\n|  Pricing Test | 22  |\n|  Marketing Email Campaign | 25  |\n|  Song Challenge | 28  |\n\nClick here to check out our site if interested i

In [ ]:
# Pretty-print the full tree.
def print_tree(nodes, indent=0):
    """Recursively print section titles, node IDs, and page numbers."""
    for node in nodes:
        prefix = "    " * indent + ("└─ " if indent > 0 else "")
        page = node.get("page_index", "?")
        node_id = node.get("node_id", "?")
        title = node.get("title", "Untitled")
        print(f"{prefix}[{node_id}] {title} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)


In [ ]:
# Count all nodes, including nested subsections.
def count_nodes(nodes):
    total = len(nodes)
    for node in nodes:
        if node.get("nodes"):
            total += count_nodes(node["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("Each node represents a section of the document.")
